# Informe Final - Segmentación Automática de Columna Vertebral y Vértebras
## Proyecto Final — Maestría en Inteligencia Artificial
### Universidad de los Andes

---

Este notebook presenta el trabajo completo del proyecto:

1. **Contexto y objetivos**
2. **Dataset: exploración**
3. **Pipeline de datos**
4. **5 arquitecturas comparadas**
5. **Carga de modelos pre-entrenados**
6. **Evaluación comparativa**
7. **Análisis per-clase (por vértebra)**
8. **Visualización de predicciones**
9. **Cálculo del ángulo de Cobb**
10. **Explicabilidad (Grad-CAM)**
11. **Conclusiones y discusión**

**IMPORTANTE:** Este notebook funciona en **CPU** para inferencia. No requiere GPU.  
Solo necesita los archivos `.pth` pre-entrenados en la carpeta `checkpoints/`.

---
## 1. Contexto y Objetivos

### Problema
La escoliosis es una deformidad tridimensional de la columna vertebral que afecta al 2-3% de la población. Su diagnóstico y seguimiento requieren la medición del **ángulo de Cobb** sobre radiografías de columna completa, un proceso que es:
- Subjetivo (alta variabilidad inter-observador)
- Lento (5-10 min por radiografía)
- Dependiente de la experiencia del radiólogo

### Solución propuesta
Sistema automatizado que:
1. **Segmenta las vértebras individuales** (C3-L5, 24 clases) usando deep learning
2. **Calcula el ángulo de Cobb** automáticamente a partir de la segmentación
3. **Explica sus predicciones** con Grad-CAM y mapas de confianza (no es caja negra)
4. **Se despliega** en servidor (Docker) y es accesible desde web o tablet

### Dataset
**MaIA Scoliosis Dataset** — 250 radiografías AP de columna completa:
- 71 pacientes normales
- 179 pacientes con escoliosis
- Máscaras multiclase (36 clases originales → 24 clínicamente relevantes)
- Ángulo de Cobb ground truth para los 179 casos de escoliosis

---
## 2. Setup e Imports

In [ ]:
import sys
from pathlib import Path

# Agregar el paquete del proyecto al path
sys.path.insert(0, str(Path('..').resolve()))

# Imports estándar
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from collections import Counter

# Imports del paquete del proyecto
from spine_segmentation.config import (
    DATASET_ROOT, DATASET_INDEX_CSV, RADIOGRAPH_METRICS_CSV,
    CHECKPOINTS_DIR, OUTPUTS_DIR, MULTICLASS_COLORS
)
from spine_segmentation.data.dataset import SpineMulticlassDataset, get_dataloaders
from spine_segmentation.data.splits import load_splits
from spine_segmentation.data.class_mapping import get_class_names, get_num_classes
from spine_segmentation.data.transforms import denormalize_image
from spine_segmentation.models.smp_models import create_model
from spine_segmentation.evaluation.metrics import (
    compute_test_metrics, compute_metrics_multiclass, print_metrics_table
)
from spine_segmentation.evaluation.visualize import plot_per_class_dice
from spine_segmentation.evaluation.cobb_angle import (
    cobb_from_binary, cobb_from_multiclass,
    evaluate_cobb_angles, plot_cobb_comparison, load_ground_truth_cobb_angles
)
from spine_segmentation.postprocessing.morphology import (
    clean_binary_mask, clean_multiclass_mask
)

# Configuración visual
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

# Detectar dispositivo (CPU o GPU)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('Ejecutando en CPU - suficiente para inferencia')

---
## 3. Exploración del Dataset

In [ ]:
# Cargar índice del dataset
df = pd.read_csv(DATASET_INDEX_CSV)

print(f'Total de muestras: {len(df)}')
print(f'\nDistribución por condición:')
print(df['split'].value_counts())
print(f'\nRatio: {df["split"].value_counts(normalize=True).round(3).to_dict()}')

# Ver primeras filas del índice
df.head(3)

In [ ]:
# Distribución de ángulos de Cobb (solo casos de escoliosis)
metrics_df = pd.read_csv(RADIOGRAPH_METRICS_CSV)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(metrics_df['cobb_angle_deg'], bins=25, color='mediumpurple', edgecolor='white')
axes[0].axvline(x=10, color='green', linestyle='--', label='10° (umbral mild)')
axes[0].axvline(x=25, color='orange', linestyle='--', label='25° (moderate)')
axes[0].axvline(x=40, color='red', linestyle='--', label='40° (severe)')
axes[0].set_xlabel('Ángulo de Cobb (grados)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de ángulos de Cobb (n=179)')
axes[0].legend()

# Distribución por severidad
cobb = metrics_df['cobb_angle_deg']
mild = ((cobb >= 10) & (cobb < 25)).sum()
moderate = ((cobb >= 25) & (cobb < 40)).sum()
severe = (cobb >= 40).sum()

axes[1].pie([mild, moderate, severe], 
            labels=[f'Mild (10-25°)\n{mild} casos', 
                    f'Moderate (25-40°)\n{moderate} casos',
                    f'Severe (>40°)\n{severe} casos'],
            colors=['#2ecc71', '#f39c12', '#e74c3c'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribución por severidad de escoliosis')

plt.suptitle('Análisis de ángulos de Cobb ground truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nEstadísticas:')
print(f'  Min: {cobb.min():.1f}°  Max: {cobb.max():.1f}°')
print(f'  Media: {cobb.mean():.1f}°  Mediana: {cobb.median():.1f}°  Std: {cobb.std():.1f}°')

In [ ]:
# Visualizar muestras del dataset con sus máscaras multiclase
sample_normal = df[df['split'] == 'Normal'].sample(2, random_state=42)
sample_scol = df[df['split'] == 'Scoliosis'].sample(4, random_state=42)
samples = pd.concat([sample_normal, sample_scol]).reset_index(drop=True)

fig, axes = plt.subplots(2, 6, figsize=(22, 11))

for i, (_, row) in enumerate(samples.iterrows()):
    # Radiografía original
    img = cv2.imread(str(DATASET_ROOT / row['radiograph_path']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"{row['image']}\n({row['split']})", fontsize=10)
    axes[0, i].axis('off')
    
    # Overlay multiclase
    mask = cv2.imread(str(DATASET_ROOT / row['multiclass_id_png']), cv2.IMREAD_UNCHANGED)
    if mask is not None:
        if mask.ndim == 3:
            mask = mask[:, :, 0]
        overlay = img.copy()
        color_mask = np.zeros_like(img)
        for cid, color in MULTICLASS_COLORS.items():
            if cid == 0:
                continue
            region = mask == cid
            if region.any():
                color_mask[region] = color
        overlay = cv2.addWeighted(overlay, 0.6, color_mask, 0.4, 0)
        axes[1, i].imshow(overlay)
    axes[1, i].set_title('Segmentación de vértebras', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Muestras del dataset: Radiografía (arriba) vs Máscara multiclase (abajo)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Pipeline de Datos

### División estratificada 70/15/15
- **Train:** 174 imágenes (49 Normal, 125 Escoliosis)
- **Validation:** 38 imágenes (11 Normal, 27 Escoliosis)
- **Test:** 38 imágenes (11 Normal, 27 Escoliosis)
- Seed fijo=42 para reproducibilidad

### Preprocesamiento
1. Resize preservando aspect ratio (lado mayor → 512px) + padding a 512×512
2. Normalización ImageNet (mean/std)
3. Para máscaras: interpolación nearest-neighbor (no corromper IDs de clase)

### Data Augmentation (solo entrenamiento)
- Affine (shift ±5%, scale ±10%, rotate ±10°)
- ElasticTransform (simula variabilidad anatómica)
- GridDistortion
- HorizontalFlip (la columna tiene simetría bilateral aproximada)
- RandomBrightnessContrast
- CLAHE (mejora contraste en radiografías)
- GaussNoise

**NOTA:** NO se usa VerticalFlip — las vértebras tienen orden anatómico estricto (C3 arriba → L5 abajo).

In [ ]:
# Cargar splits (fijos, reproducibles)
splits = load_splits()

print('Distribución por split:')
for split_name, images in splits.items():
    split_df = df[df['image'].isin(images)]
    counts = split_df['split'].value_counts()
    print(f'  {split_name:>6}: {len(images):3d} imagenes | '
          f'Normal={counts.get("Normal", 0):2d} | '
          f'Scoliosis={counts.get("Scoliosis", 0):3d}')

---
## 5. Arquitecturas Comparadas

Se comparan **5 arquitecturas** representando distintos paradigmas de segmentación semántica:

| # | Modelo | Paradigma | Justificación |
|---|--------|-----------|---------------|
| 1 | **U-Net + ResNet50** | CNN clásica | Baseline biomédico estándar. Skip connections en 5 niveles. |
| 2 | **U-Net + EfficientNet-B4** | CNN eficiente | Compound scaling. Mejor para deploy en tablet/edge (19 MB INT8). |
| 3 | **DeepLabV3+ + ResNet50** | CNN multi-escala | ASPP captura vértebras de tamaño variable (cervicales vs lumbares). |
| 4 | **U-Net + MiT-B3** | Transformer (SegFormer) | Self-attention global. Maneja oclusión vertebral por rotación. |
| 5 | **MAnet + MiT-B5** | Transformer + atención dual | Encoder transformer + decoder con Position Attention Module. |

**Narrativa:** CNN clásica → CNN eficiente → CNN multi-escala → Transformer → Transformer + atención dual.  
Progresión de capacidad para demostrar el aporte de los mecanismos de atención.

---
## 6. Carga de Modelos Pre-entrenados

**Esta sección NO requiere GPU** — carga pesos y corre en CPU.

Los pesos están en la carpeta `checkpoints/`. Si no los tienen, solicítenselos al líder del equipo (compartidos vía OneDrive/Drive).

In [ ]:
def load_trained_model(model_name, task='multiclass', num_classes=24, device='cpu'):
    """Carga un modelo con pesos pre-entrenados (funciona en CPU)."""
    ckpt_path = CHECKPOINTS_DIR / f'{model_name}_{task}_best.pth'
    
    if not ckpt_path.exists():
        print(f'  [X] No existe: {ckpt_path.name}')
        return None, None
    
    model = create_model(model_name, num_classes=num_classes)
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    return model, checkpoint

# Listar checkpoints disponibles
print('Checkpoints disponibles en', CHECKPOINTS_DIR)
print('-' * 60)
for f in sorted(CHECKPOINTS_DIR.glob('*.pth')):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name:<50} {size_mb:>6.1f} MB')

In [ ]:
# Cargar los 5 modelos multiclase
MODELS_TO_LOAD = [
    'unet_resnet50',
    'unet_efficientnet_b4', 
    'deeplabv3plus_resnet50',
    'unet_mit_b3',
    'manet_mit_b5',
]

modelos = {}
checkpoints_info = {}

for name in MODELS_TO_LOAD:
    print(f'Cargando {name}...')
    model, ckpt = load_trained_model(name, task='multiclass', num_classes=24, device=DEVICE)
    if model is not None:
        modelos[name] = model
        checkpoints_info[name] = {
            'best_val_dice': ckpt.get('best_val_dice', 'N/A'),
            'epoch': ckpt.get('epoch', 'N/A'),
        }
        best_dice = ckpt.get('best_val_dice', 0)
        epoch = ckpt.get('epoch', 0)
        print(f'  [OK] Dice val={best_dice:.4f} (epoca {epoch + 1})')
    print()

print(f'Total modelos cargados: {len(modelos)}/5')

---
## 7. Evaluación Comparativa en Test Set

In [ ]:
# DataLoader de test (multiclase)
loaders = get_dataloaders(task='multiclass', scheme='vertebrae_24', splits_dict=splits)
test_loader = loaders['test']

print(f'Test set: {len(test_loader.dataset)} imagenes')
print(f'Batches: {len(test_loader)}')

In [ ]:
# Evaluar cada modelo en el test set
resultados_finales = []
per_class_dice_all = {}

for name, model in modelos.items():
    print(f'\n{"="*70}')
    print(f'Evaluando: {name}')
    print(f'{"="*70}')
    
    metrics = compute_test_metrics(
        model, test_loader,
        task='multiclass', num_classes=24,
        device=DEVICE,
    )
    
    print(f'  Dice mean:   {metrics["test_dice_mean"]:.4f}')
    print(f'  IoU mean:    {metrics["test_iou_mean"]:.4f}')
    print(f'  Pixel Acc:   {metrics["test_pixel_acc"]:.4f}')
    
    resultados_finales.append({
        'Modelo': name,
        'Paradigma': 'Transformer' if 'mit' in name else 'CNN',
        'Dice': metrics['test_dice_mean'],
        'IoU': metrics['test_iou_mean'],
        'PixAcc': metrics['test_pixel_acc'],
    })
    
    per_class_dice_all[name] = metrics.get('per_class_dice', [])

# Tabla comparativa ordenada
df_results = pd.DataFrame(resultados_finales).sort_values('Dice', ascending=False).reset_index(drop=True)
df_results.index += 1
df_results.index.name = 'Ranking'

print(f'\n{"="*70}')
print('TABLA COMPARATIVA FINAL (ordenada por Dice descendente)')
print(f'{"="*70}')
df_results.style.format({'Dice': '{:.4f}', 'IoU': '{:.4f}', 'PixAcc': '{:.4f}'})

In [ ]:
# Visualizar comparación como bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = ['Dice', 'IoU', 'PixAcc']
for i, metric in enumerate(metrics_to_plot):
    df_sorted = df_results.sort_values(metric, ascending=True)
    colors = ['#3498db' if p == 'CNN' else '#e74c3c' for p in df_sorted['Paradigma']]
    axes[i].barh(df_sorted['Modelo'], df_sorted[metric], color=colors)
    axes[i].set_xlabel(metric, fontsize=12)
    axes[i].set_title(f'Test {metric}', fontsize=13, fontweight='bold')
    
    # Añadir valores en las barras
    for j, v in enumerate(df_sorted[metric]):
        axes[i].text(v, j, f'  {v:.4f}', va='center', fontsize=9)

# Leyenda
import matplotlib.patches as mpatches
cnn_patch = mpatches.Patch(color='#3498db', label='CNN')
trans_patch = mpatches.Patch(color='#e74c3c', label='Transformer')
fig.legend(handles=[cnn_patch, trans_patch], loc='upper right', fontsize=11)

plt.suptitle('Comparativa de los 5 modelos multiclase', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Análisis Per-Clase (por vértebra)

In [ ]:
# Gráfico per-class Dice del mejor modelo
class_names = get_class_names('vertebrae_24')

best_model_name = df_results.iloc[0]['Modelo']
best_dice_per_class = per_class_dice_all[best_model_name]

print(f'Análisis per-clase del mejor modelo: {best_model_name}')

# Build DataFrame con las 22 vértebras
vert_data = []
for c in range(1, 23):
    name = class_names.get(c, f'C{c}')
    region = 'Cervical' if name.startswith('C') else 'Torácica' if name.startswith('T') else 'Lumbar'
    vert_data.append({'Vertebra': name, 'Region': region, 'Dice': best_dice_per_class[c] if c < len(best_dice_per_class) else 0})

df_vert = pd.DataFrame(vert_data)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
colors_region = {'Cervical': '#e74c3c', 'Torácica': '#2ecc71', 'Lumbar': '#3498db'}
colors = [colors_region[r] for r in df_vert['Region']]

bars = ax.bar(df_vert['Vertebra'], df_vert['Dice'], color=colors, edgecolor='white')
ax.set_ylabel('Dice Coefficient', fontsize=12)
ax.set_xlabel('Vértebra', fontsize=12)
ax.set_title(f'Dice por vértebra — {best_model_name}', fontsize=14, fontweight='bold')
ax.axhline(y=df_vert['Dice'].mean(), color='black', linestyle='--', label=f'Media: {df_vert["Dice"].mean():.3f}')
ax.legend([
    mpatches.Patch(color='#e74c3c', label='Cervicales (C3-C7)'),
    mpatches.Patch(color='#2ecc71', label='Torácicas (T1-T12)'),
    mpatches.Patch(color='#3498db', label='Lumbares (L1-L5)'),
], loc='upper left')
plt.xticks(rotation=45)

# Valores encima de cada barra
for bar, dice in zip(bars, df_vert['Dice']):
    if dice > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{dice:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/dice_por_vertebra.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla por región
print('\nDice promedio por región anatómica:')
print(df_vert.groupby('Region')['Dice'].agg(['mean', 'std', 'min', 'max']).round(4))

---
## 9. Visualización de Predicciones

In [ ]:
# Comparar predicciones de los 5 modelos en la misma muestra
test_ds = SpineMulticlassDataset(split='test', scheme='vertebrae_24', splits_dict=splits)

# Seleccionar 1 muestra representativa (caso de escoliosis)
sample_idx = 5  # puedes cambiar para ver otras
sample = test_ds[sample_idx]
img_tensor = sample['image'].unsqueeze(0).to(DEVICE)
mask_gt = sample['mask'].numpy()
img_vis = denormalize_image(sample['image'])

# Predicciones de cada modelo
predictions = {}
with torch.no_grad():
    for name, model in modelos.items():
        logits = model(img_tensor)
        pred_mask = logits.argmax(dim=1).cpu().numpy()[0].astype(np.uint8)
        pred_clean = clean_multiclass_mask(pred_mask, num_classes=24)
        predictions[name] = pred_clean

# Plot comparativo
n_models = len(predictions)
fig, axes = plt.subplots(2, n_models + 1, figsize=(4 * (n_models + 1), 8))

# Row 1: raw masks
axes[0, 0].imshow(img_vis)
axes[0, 0].set_title(f"Original\n{sample['image_name']}", fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

axes[1, 0].imshow(mask_gt, cmap='nipy_spectral', vmin=0, vmax=23)
axes[1, 0].set_title('Ground Truth', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

for i, (name, pred) in enumerate(predictions.items(), start=1):
    # Overlay
    overlay = img_vis.copy()
    color_mask = np.zeros_like(overlay)
    for cid, color in MULTICLASS_COLORS.items():
        if cid == 0:
            continue
        color_mask[pred == cid] = color
    overlay = cv2.addWeighted(overlay, 0.6, color_mask, 0.4, 0)
    
    axes[0, i].imshow(overlay)
    axes[0, i].set_title(name, fontsize=10, fontweight='bold')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(pred, cmap='nipy_spectral', vmin=0, vmax=23)
    axes[1, i].set_title('Predicción', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle(f'Comparativa de predicciones — {sample["image_name"]} ({sample["condition"]})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/comparativa_predicciones.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Cálculo del Ángulo de Cobb

Se implementan **dos métodos**:

**Método A — Esqueletización + B-spline (replica del semestre anterior):**
1. Segmentación binaria de la columna
2. Esqueletización → eliminar ramas
3. Ajuste de B-spline
4. Derivadas primera y segunda
5. Puntos de inflexión (cambios de signo de la segunda derivada)
6. Ángulo entre tangentes en los puntos de inflexión

**Método B — Orientación de placas vertebrales (contribución nueva):**
1. Segmentación multiclase (identifica cada vértebra)
2. Para cada vértebra: calcular centroide y orientación (PCA)
3. Identificar vértebras terminales de la curva
4. Cobb = ángulo entre las placas terminales

El **Método B es el clínicamente estándar** — lo que hace el radiólogo manualmente.

In [ ]:
# Cargar ground truth de Cobb
gt_cobb = load_ground_truth_cobb_angles()
print(f'Ground truth de Cobb: {len(gt_cobb)} casos')

# Calcular Cobb con el mejor modelo (multiclase)
best_model = modelos[best_model_name]

cobb_predictions = {}
print(f'\nCalculando angulo de Cobb con {best_model_name} (Metodo B - Multiclase)...')

for i in range(len(test_ds)):
    sample = test_ds[i]
    name = sample['image_name']
    
    # Solo casos de escoliosis tienen Cobb en ground truth
    if not name.startswith('S_'):
        continue
    
    img_t = sample['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = best_model(img_t)
    
    pred = logits.argmax(dim=1).cpu().numpy()[0].astype(np.uint8)
    pred_clean = clean_multiclass_mask(pred, num_classes=24)
    
    result = cobb_from_multiclass(pred_clean)
    if result['success'] and result['cobb_angle_deg'] is not None:
        cobb_predictions[name] = result['cobb_angle_deg']

print(f'Predicciones generadas: {len(cobb_predictions)}')

In [ ]:
# Evaluar contra ground truth
if cobb_predictions:
    eval_cobb = evaluate_cobb_angles(cobb_predictions)
    
    print(f'\n{"="*60}')
    print(f'RESULTADOS ANGULO DE COBB — {best_model_name}')
    print(f'{"="*60}')
    print(f'  N samples evaluadas:  {eval_cobb["n_samples"]}')
    print(f'  MAE (grados):         {eval_cobb["mae_deg"]:.1f}°')
    print(f'  Mediana error:        {eval_cobb["median_error_deg"]:.1f}°')
    print(f'  Error máximo:         {eval_cobb["max_error_deg"]:.1f}°')
    print(f'  Correlación Pearson:  {eval_cobb.get("correlation", 0):.3f}')
    print(f'  Dentro de 5°:         {eval_cobb.get("within_5_deg", 0)*100:.1f}%')
    print(f'  Dentro de 10°:        {eval_cobb.get("within_10_deg", 0)*100:.1f}%')
    
    # Bland-Altman + scatter
    plot_cobb_comparison(
        eval_cobb,
        save_path='../outputs/cobb_comparison.png',
        title=f'Ángulo de Cobb — {best_model_name}'
    )
else:
    print('No se generaron predicciones de Cobb.')

---
## 11. Explicabilidad (Grad-CAM + Mapas de Confianza)

**No es caja negra.** El médico puede ver:
- **Grad-CAM:** qué regiones influyeron en la decisión
- **Mapa de confianza:** dónde el modelo está seguro (verde) vs inseguro (rojo)
- **Reporte clínico:** vértebras detectadas, ángulo de Cobb, severidad

Esto es requerido por regulación médica (FDA, CE marking) para AI de apoyo al diagnóstico.

In [ ]:
from spine_segmentation.evaluation.explainability import (
    generate_gradcam, generate_confidence_map, generate_explanation_panel
)

# Generar panel de explicabilidad para 3 muestras con el mejor modelo
output_dir = Path('../outputs/explanations')
output_dir.mkdir(parents=True, exist_ok=True)

for idx in [3, 10, 20]:  # 3 muestras
    if idx >= len(test_ds):
        continue
    sample = test_ds[idx]
    img_t = sample['image'].unsqueeze(0).to(DEVICE)
    img_vis = denormalize_image(sample['image'])
    
    save_path = str(output_dir / f'explain_{sample["image_name"].replace(".jpg",".png")}')
    
    try:
        generate_explanation_panel(
            model=best_model,
            input_tensor=img_t,
            original_image=img_vis,
            model_name=best_model_name,
            task='multiclass',
            save_path=save_path,
            image_name=sample['image_name']
        )
        print(f'[OK] Panel generado: {save_path}')
    except Exception as e:
        print(f'[!] Error en {sample["image_name"]}: {e}')

print(f'\nPaneles guardados en: {output_dir}')

---
## 12. Conclusiones y Discusión

### Hallazgos principales

#### 1. Los transformers superaron a las CNNs
Contrario a la hipótesis inicial (que decía que con solo 174 imágenes de entrenamiento los transformers no tendrían suficientes datos), **los modelos con self-attention global rindieron mejor**:

| Ranking | Modelo | Paradigma | Dice |
|---------|--------|-----------|------|
| 🥇 | MAnet + MiT-B5 | Transformer + atención dual | **0.3271** |
| 🥈 | U-Net + MiT-B3 | Transformer | 0.3157 |
| 🥉 | U-Net + ResNet50 | CNN | 0.2691 |

**Interpretación:** El self-attention captura relaciones globales entre vértebras distantes (ej. C3 y L5) que las CNNs puras con campo receptivo local no pueden modelar eficientemente.

#### 2. El dataset es pequeño para este problema
Dice promedio ~0.30 es modesto. Posibles mejoras:
- Más datos (data augmentation más agresiva, datasets públicos complementarios)
- Pre-entrenamiento en datasets médicos (RadImageNet)
- Self-supervised pre-training específico de radiografías

#### 3. Desbalance extremo de clases
- Background: 95.9% de píxeles
- Cervicales (C3-C5): < 0.01% de píxeles
- Las vértebras cervicales son las más difíciles de segmentar

#### 4. Desafío de la oclusión en escoliosis severa
Las vértebras rotadas parcialmente ocultas son el problema más difícil. Los transformers mitigan esto usando el contexto de las vértebras vecinas.

### Limitaciones
- Dataset privado (250 imágenes) no comparable con estado del arte en datasets públicos
- Split de test pequeño (38 imágenes) → variabilidad alta en métricas
- Cálculo de Cobb depende de la calidad de la segmentación (error acumulativo)

### Trabajo futuro
- Data augmentation 3D / aprendizaje semi-supervisado
- Quantización INT8 para deploy en tablet (modelo 4x más ligero)
- Combinar ensemble de los 5 modelos
- Validación clínica con radiólogos (inter-observer agreement)
- Extender a segmentación de discos intervertebrales

---
## 13. Para el Equipo

### Cómo ejecutar este notebook (sin GPU)
1. Instalar dependencias: `pip install -r requirements.txt`
2. Obtener pesos pre-entrenados (compartidos vía OneDrive) → colocar en `checkpoints/`
3. Obtener dataset (solicitar al profesor) → colocar en `MaIA_Scoliosis_Dataset/`
4. Abrir este notebook y ejecutar todas las celdas (menú Cell → Run All)

### Cómo usar la app web
```bash
python -m spine_segmentation.deployment.app
```
Abrir `http://localhost:7860` en el navegador.

### Contacto
Para dudas sobre el proyecto o solicitud de pesos: **elvis.hernandez@en-firme.com**

### Disclaimer clínico
Esta herramienta es de **APOYO al diagnóstico**. NO reemplaza el criterio del profesional de la salud. Toda decisión clínica debe ser validada por un especialista calificado.